In [1]:
import os
import numpy as np
import pandas as pd
import torch
from scipy.special import softmax
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from netcal.scaling import TemperatureScaling
from netcal.metrics import ECE

set_seed(42)

MODEL_NAME = "csebuetnlp/banglabert"
DATA_DIR = "../data/processed/sentiment"
OUTPUT_DIR = "../models/mvp_banglabert_sentiment"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs("../results/mvp", exist_ok=True)

def load_subset(split_name: str, n_samples: int) -> pd.DataFrame:
    df = pd.read_csv(f"{DATA_DIR}/{split_name}.csv")
    df["labels"] = df["labels"].astype(int)
    return df.groupby("labels", group_keys=False).apply(
        lambda x: x.sample(min(len(x), n_samples // 3), random_state=42)
    ).reset_index(drop=True)

train_df = load_subset("train", 150)
val_df = load_subset("val", 60)
cal_df = load_subset("cal", 60)
test_df = load_subset("test", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text_clean"], padding="max_length", truncation=True, max_length=128)

def to_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(df[["text_clean", "labels"]])
    ds = ds.map(tokenize_fn, batched=True, remove_columns=["text_clean"])
    ds.set_format("torch")
    return ds

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
cal_ds = to_hf_dataset(cal_df)
test_ds = to_hf_dataset(test_df)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    seed=42,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

trainer.train()

cal_logits = trainer.predict(cal_ds).predictions
test_logits = trainer.predict(test_ds).predictions

cal_probs = softmax(cal_logits, axis=1)
test_probs = softmax(test_logits, axis=1)

cal_labels = cal_df["labels"].values
test_labels = test_df["labels"].values

ts = TemperatureScaling()
ts.fit(cal_probs, cal_labels)
cal_probs_scaled = ts.transform(cal_probs)
test_probs_scaled = ts.transform(test_probs)

ece_metric = ECE(bins=15)
ece_before = ece_metric.measure(test_probs, test_labels)
ece_after = ece_metric.measure(test_probs_scaled, test_labels)
macro_f1 = f1_score(test_labels, np.argmax(test_probs_scaled, axis=1), average="macro")

alpha = 0.10
n_cal = len(cal_labels)
cal_scores = 1.0 - cal_probs_scaled[np.arange(n_cal), cal_labels]
q_level = min(np.ceil((n_cal + 1) * (1.0 - alpha)) / n_cal, 1.0)
q_hat = np.quantile(cal_scores, q_level, method="higher")

pred_sets = test_probs_scaled >= (1.0 - q_hat)
coverage = np.mean(pred_sets[np.arange(len(test_labels)), test_labels])
avg_set_size = np.mean(np.sum(pred_sets, axis=1))

mvp_summary = pd.DataFrame([{
    "macro_f1": round(float(macro_f1), 4),
    "ece_uncalibrated": round(float(ece_before), 4),
    "ece_calibrated": round(float(ece_after), 4),
    "conformal_coverage_90": round(float(coverage), 4),
    "avg_prediction_set_size": round(float(avg_set_size), 4)
}])

mvp_summary.to_csv("../results/mvp/mvp_validation_metrics.csv", index=False)
print(mvp_summary.to_string(index=False))

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24280\1640966618.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("labels", group_keys=False).apply(
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24280\1640966618.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("labels", group_keys=False).apply(
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24280\1640966618.py:29: Depre

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

d:\Thesis WorkFlow\bangla-calibration\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--csebuetnlp--banglabert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

{'eval_loss': 1.0900988578796387, 'eval_runtime': 9.1262, 'eval_samples_per_second': 6.575, 'eval_steps_per_second': 0.438, 'epoch': 1.0}
{'train_runtime': 84.0718, 'train_samples_per_second': 1.784, 'train_steps_per_second': 0.119, 'train_loss': 1.0964421272277831, 'epoch': 1.0}


  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

 macro_f1  ece_uncalibrated  ece_calibrated  conformal_coverage_90  avg_prediction_set_size
   0.3329            0.0136          0.0136                 0.9667                     2.65


In [2]:
shift_files = {
    "D0_clean": "test.csv",
    "D2_codemixed": "d2_codemixed_test.csv",
    "D3_noise_05pct": "synthetic_noise_05pct.csv",
    "D3_noise_10pct": "synthetic_noise_10pct.csv",
    "D3_noise_20pct": "synthetic_noise_20pct.csv"
}

shift_results = []

for condition, filename in shift_files.items():
    df_cond = pd.read_csv(f"{DATA_DIR}/{filename}")
    df_cond["labels"] = df_cond["labels"].astype(int)
    df_sub = df_cond.groupby("labels", group_keys=False).apply(
        lambda x: x.sample(min(len(x), 20), random_state=42)
    ).reset_index(drop=True)
    
    ds_sub = to_hf_dataset(df_sub)
    logits = trainer.predict(ds_sub).predictions
    probs = softmax(logits, axis=1)
    probs_scaled = ts.transform(probs)
    labels = df_sub["labels"].values
    
    f1 = f1_score(labels, np.argmax(probs_scaled, axis=1), average="macro")
    ece_raw = ece_metric.measure(probs, labels)
    ece_cal = ece_metric.measure(probs_scaled, labels)
    
    pred_sets = probs_scaled >= (1.0 - q_hat)
    cov = np.mean(pred_sets[np.arange(len(labels)), labels])
    set_size = np.mean(np.sum(pred_sets, axis=1))
    
    shift_results.append({
        "condition": condition,
        "macro_f1": round(float(f1), 4),
        "ece_uncalibrated": round(float(ece_raw), 4),
        "ece_calibrated": round(float(ece_cal), 4),
        "conformal_coverage_90": round(float(cov), 4),
        "avg_set_size": round(float(set_size), 4)
    })

shift_summary_df = pd.DataFrame(shift_results)
shift_summary_df.to_csv("../results/mvp/mvp_shift_validation.csv", index=False)
print(shift_summary_df.to_string(index=False))

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24280\3047436084.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sub = df_cond.groupby("labels", group_keys=False).apply(


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

  0%|          | 0/4 [00:00<?, ?it/s]

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24280\3047436084.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sub = df_cond.groupby("labels", group_keys=False).apply(


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24280\3047436084.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sub = df_cond.groupby("labels", group_keys=False).apply(


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

  0%|          | 0/4 [00:00<?, ?it/s]

     condition  macro_f1  ece_uncalibrated  ece_calibrated  conformal_coverage_90  avg_set_size
      D0_clean    0.3329            0.0136          0.0136                 0.9667        2.6500
  D2_codemixed    0.1878            0.0390          0.0390                 0.6833        2.1667
D3_noise_05pct    0.4300            0.0839          0.0839                 1.0000        2.7667
D3_noise_10pct    0.3989            0.0535          0.0535                 1.0000        2.8667
D3_noise_20pct    0.2473            0.0299          0.0299                 1.0000        2.9500
